In [2]:
import kagglehub
from pathlib import Path
import math
import calendar
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from scipy import stats
from scipy.stats import pointbiserialr, f_oneway, chi2_contingency
from sklearn import metrics
from sklearn.metrics import make_scorer
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import ElasticNet

c:\Users\rk1523\Desktop\Projects\weather-forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset_handle = "nelgiriyewithana/global-weather-repository"
data_folder = Path("Data-main")

kagglehub.dataset_download(
    dataset_handle,
    output_dir=data_folder,
    force_download=True
)

100%|██████████| 12.0M/12.0M [00:01<00:00, 6.53MB/s]

Extracting files...


WindowsPath('Data-main')

In [64]:
csv_path = list(data_folder.rglob("*.csv"))[0]
df = pd.read_csv(csv_path)
display(df.head())
print(df.shape)
print(df.columns)

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


(151632, 41)
Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination'],
      dtype='str')


In [65]:
missing_data = df.isna().sum()
print(missing_data[missing_data > 0])

Series([], dtype: int64)


In [66]:
print(sum(pd.Categorical(df["country"]).value_counts() < 10))
print(sum(pd.Categorical(df["location_name"]).value_counts() < 10))

25
36



# Data feature engineering
Goal: predicting temperature_celsius and through spatiotemporal time series analysis

### Dropped duplicate features: 
- temperature_fahrenheit
- wind_mph
- pressure_in
- precip_in
- visibility_miles
- gust_mph
- feels_like_fahrenheit
- air_quality_us-epa-index
- air_quality_gb-defra-index

These features are either a direct duplicate with non-metric units, or a combination of other features (e.g. air quality index is a combination of the more granular particular measurements). Removal ensures correlated features do not 

### Dropped leakage-prone features:
- feels_like_celsius

This direct contains information about temperature, so inclusion can cause data leakage and overfitting. 

### Kept features: 
- humidity
- cloud
- visibility_km
- uv_index
- gust_kph
- pressure_mb

also dropped "condition_text" as there were 58 categories.

In [75]:
# Copy original dataframe
weather_df = df.copy()

features_to_remove = ['country', 'location_name', 'timezone',
                     'condition_text',
                     'last_updated_epoch',
                     'temperature_fahrenheit',
                     'wind_mph','wind_direction', 
                     'pressure_in', 'precip_in',
                     'feels_like_celsius','feels_like_fahrenheit',
                     'visibility_miles',
                     'gust_mph',
                     'air_quality_us-epa-index', 'air_quality_gb-defra-index', 
                     'moonrise', 'moonset', 'moon_phase', 'moon_illumination']

weather_df.drop(columns = features_to_remove, inplace=True)

### Modified time features:
- last_updated_epoch
- last_updated
- sunrise
- sunset
- moonrise
- moonset
- moon_phase
- moon_illumination

modified to: 
- last_updated (in datetime)
- hour_sin
- hour_cos
- day_sin
- day_cos
- sunrise
- sunset

This encodes the most relevant features for temperature prediction. The hours of day in sin and cos represent the cyclical nature of the 24 hour clock and ensure that 23:59 and 0:01 are proximally encoded. This is the same with day sin and day cos. 

In [76]:
# Update time to datetime format
weather_df["last_updated"] = pd.to_datetime(weather_df["last_updated"], errors="raise")

# Extract date and year to calculate progression through the year as a fraction
date = weather_df["last_updated"].dt
day_of_year = date.dayofyear
annual_progress = (day_of_year - 1) / np.where(date.is_leap_year, 366, 365)

# Convert fraction to angle in radians
day_angle = 2 * np.pi * annual_progress

# Deconstruct angle into sin and cos components
weather_df = weather_df.assign(
    day_sin=np.sin(day_angle),
    day_cos=np.cos(day_angle),
)

# Extract time of day in hours and calculate progression through the day as a fraction
time_of_day = (
    weather_df["last_updated"].dt.hour
    + weather_df["last_updated"].dt.minute / 60
    + weather_df["last_updated"].dt.second / 3600
)
daily_progress = time_of_day / 24

# Convert fraction to angle in radians
time_angle = 2 * np.pi * daily_progress

# Deconstruct angle into sin and cos components
weather_df = weather_df.assign(
    time_sin=np.sin(time_angle),
    time_cos=np.cos(time_angle),
)

# Convert sunrise and sunset to datetime format
sunrise = pd.to_datetime(
    weather_df["sunrise"],
    format="%I:%M %p",
    errors="coerce"
)
sunset = pd.to_datetime(
    weather_df["sunset"],
    format="%I:%M %p",
    errors="coerce"
)

# Convert to decimal values and assign
weather_df = weather_df.assign(
    sunrise=sunrise.dt.hour + sunrise.dt.minute / 60,
    sunset=sunset.dt.hour + sunset.dt.minute / 60
)

### Modified geographical features: 
- latitude
- longitude
- timezone
- country
- location_name

reduced to: 
- geo_x
- geo_y
- geo_z

This will encode the geographical information numerically as spherical coordinates. 


### Modified wind features: 
- wind_kph
- wind_direction
- wind_degree

modified to: 
- wind_x
- wind_y

Encodes all information about wind speed and degree in one vector, also capturing the cyclical nature of the degree feature. Wind direction is categorical and redundant and is hence removed. 

In [77]:
latitude_radians = np.radians(weather_df["latitude"])
longitude_radians = np.radians(weather_df["longitude"])

weather_df["geo_x"] = np.cos(latitude_radians) * np.cos(longitude_radians)
weather_df["geo_y"] = np.cos(latitude_radians) * np.sin(longitude_radians)
weather_df["geo_z"] = np.sin(latitude_radians)

weather_df.drop(columns = ["latitude", "longitude"], inplace=True)

In [78]:
# Convert wind direction from degrees to radians
wind_rad = np.radians(weather_df["wind_degree"])

# Meteorological convention:
# wind_degree = direction wind comes FROM
weather_df["wind_x"] = -weather_df["wind_kph"] * np.sin(wind_rad)  # east-west component
weather_df["wind_y"] = -weather_df["wind_kph"] * np.cos(wind_rad)  # north-south component

weather_df.drop(columns = ["wind_degree", "wind_kph"], inplace=True)

In [79]:
weather_df.set_index("last_updated")

,temperature_celsius,pressure_mb,precip_mm,humidity,cloud,visibility_km,uv_index,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,...,sunset,day_sin,day_cos,time_sin,time_cos,geo_x,geo_y,geo_z,wind_x,wind_y
last_updated,,,,,,,,,,,,,,,,,,,,,
2024-05-16 13:15:00,26.6,1012.0,0.00,24,30,10.0,7.0,15.3,277.0,103.0,...,18.833333,0.722117,-0.691771,-0.321439,-0.946930,0.292852,0.770127,0.566694,4.982268,-1.233155e+01
2024-05-16 10:45:00,19.0,1012.0,0.10,94,75,10.0,5.0,18.4,193.6,97.3,...,19.900000,0.722117,-0.691771,0.321439,-0.946930,0.706436,0.254611,0.660395,7.199221,-8.579698e+00
2024-05-16 09:45:00,23.0,1011.0,0.00,29,0,10.0,5.0,22.3,540.7,12.2,...,19.833333,0.722117,-0.691771,0.555570,-0.831470,0.800015,0.042627,0.598464,14.870597,-2.622087e+00
2024-05-16 10:45:00,6.3,1007.0,0.30,61,100,2.0,2.0,13.7,170.2,64.4,...,21.183333,0.722117,-0.691771,0.321439,-0.946930,0.737018,0.019557,0.675590,6.825560,9.747909e+00
2024-05-16 09:45:00,26.0,1011.0,0.00,89,50,10.0,8.0,20.2,2964.0,19.0,...,17.916667,0.722117,-0.691771,0.555570,-0.831470,0.961896,0.226142,-0.153676,-6.500000,1.125833e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-06 02:00:00,17.3,1017.0,0.00,93,39,10.0,0.0,14.4,181.0,28.0,...,18.883333,-0.060213,-0.998186,0.500000,0.866025,0.385504,-0.904531,0.182236,-1.759970,6.568296e+00
2026-07-06 13:00:00,31.2,1001.0,0.04,79,0,10.0,10.0,10.4,517.0,142.0,...,18.700000,-0.060213,-0.998186,-0.258819,-0.965926,-0.254922,0.897885,0.358910,7.461338,5.032736e+00
2026-07-06 08:45:00,23.9,1013.0,0.00,28,43,10.0,2.7,10.8,119.0,88.0,...,18.633333,-0.060213,-0.998186,0.751840,-0.659346,0.691242,0.672361,0.264794,3.460542,9.922945e-01
